# Sensitivitätsanalyse

In [15]:
import os
import papermill as pm
import pandas as pd
import numpy as np  

# ==========================================
# Anzahl der Modelle zwischen upper und lower bound
# num_runs = 5 -> 5 Schritte inkl. lower & upper bound (3 Zwischenwerte)
num_runs = 2
# ==========================================

# Werte und Grenzen aus Excel einlesen
df = pd.read_excel("Sensitivitäten.xlsx", sheet_name="Datenauswahl", decimal=',')
df.set_index("Variable", inplace=True)

# Struktur für Ergebnis laden
df_mc = pd.read_excel("Sensitivitäten.xlsx", sheet_name="Ergebnisse", engine="openpyxl")

# Modellpfade
model_dir = os.path.abspath(os.path.join("..", "Südafrika_Methanol_1_DAC_Greenfield"))
notebook_path = os.path.join(model_dir, "Modell_Methanol_DAC_Greenfield_Südafrika.ipynb")

#Papermill schreibt das Ergebnis-Notebook dorthin
trash_path = "trash_output.ipynb"

# Laufzähler über alle Variablen hinweg
run_counter = 0

# Schleife über alle Variablen (Zeilen in df)
for var in df.index:
    low = df.loc[var, "lower bound"]
    high = df.loc[var, "upper bound"]

    # Linear interpolierte Werte zwischen low und high (inkl. Grenzen)
    values = np.linspace(low, high, num_runs)

    print(f"Variable: {var} ({low} → {high}, {num_runs} Schritte)")

    # Für diese Variable num_runs Modelle rechnen
    for value in values:
        run_counter += 1
        run_name = f"Run_{run_counter}"
        print(f"Starte {run_name} für Variable '{var}' mit Wert {value:.5f}")

        # overrides: nur EIN Wert (eine Variable)
        overrides = {var: round(float(value), 5)}

        # Notebook ausführen mit overrides
        pm.execute_notebook(
            notebook_path,
            output_path=trash_path,
            parameters={"overrides": overrides},
            cwd=model_dir
        )
        os.remove(trash_path)

        # Ergebnis einlesen
        ergebnisse_path = os.path.join(model_dir, "Ergebnisse.csv")
        Ergebnisse_df = pd.read_csv(ergebnisse_path, sep=';', decimal=',')
        Ergebnisse_df.set_index("Variable", inplace=True)
        Ergebnisse = Ergebnisse_df["Wert"].to_dict()

        # Neue Spalte in df_mc anlegen
        if run_name not in df_mc.columns:
            df_mc[run_name] = None

        # Input (übersteuerte Variable) eintragen
        for v, val in overrides.items():
            df_mc.loc[df_mc["Variable"] == v, run_name] = val

        # Outputs eintragen
        for v, val in Ergebnisse.items():
            df_mc.loc[df_mc["Variable"] == v, run_name] = val

        # Zahlen erzwingen
        df_mc[run_name] = pd.to_numeric(df_mc[run_name], errors="coerce")

        print(f"{run_name} abgeschlossen.")
        
        # NEU: nach jedem Lauf zwischenspeichern
        df_mc.to_csv("Sensitivität_Ergebnisse.csv", sep=";", encoding="utf-8-sig", decimal=',', index=False)


# Finale Ausgabe
df_mc.to_csv("Sensitivität_Ergebnisse.csv", sep=";", encoding="utf-8-sig", decimal=',', index=False)
print("Export abgeschlossen → Sensitivität_Ergebnisse.csv")


Variable: capex_pem (1000.0 → 3000.0, 2 Schritte)
Starte Run_1 für Variable 'capex_pem' mit Wert 1000.00000


Executing: 100%|██████████| 154/154 [01:34<00:00,  1.64cell/s]


Run_1 abgeschlossen.
Starte Run_2 für Variable 'capex_pem' mit Wert 3000.00000


Executing: 100%|██████████| 154/154 [01:38<00:00,  1.56cell/s]


Run_2 abgeschlossen.
Variable: capex_wkaon (800.0 → 1600.0, 2 Schritte)
Starte Run_3 für Variable 'capex_wkaon' mit Wert 800.00000


...

Run_4 abgeschlossen.
Variable: wacc_change (0.008 → 0.048, 2 Schritte)
Starte Run_5 für Variable 'wacc_change' mit Wert 0.00800


Executing: 100%|██████████| 154/154 [01:13<00:00,  2.09cell/s]


Run_5 abgeschlossen.
Starte Run_6 für Variable 'wacc_change' mit Wert 0.04800


Executing: 100%|██████████| 154/154 [01:24<00:00,  1.82cell/s]


Run_6 abgeschlossen.
Export abgeschlossen → Sensitivität_Ergebnisse.csv
